# Lab | Pandas

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

The data includes information such as customer ID, state, gender, education, income, and other variables that can be used to perform various analyses.

Throughout the lab, we will be using the pandas library in Python to manipulate and analyze the data. Pandas is a powerful library that provides various data manipulation and analysis tools, including the ability to load and manipulate data from a variety of sources, including CSV files.

### Data Description

- Customer - Customer ID

- ST - State where customers live

- Gender - Gender of the customer

- Education - Background education of customers 

- Customer Lifetime Value - Customer lifetime value(CLV) is the total revenue the client will derive from their entire relationship with a customer. In other words, is the predicted or calculated value of a customer over their entire duration as a policyholder with the insurance company. It is an estimation of the net profit that the insurance company expects to generate from a customer throughout their relationship with the company. Customer Lifetime Value takes into account factors such as the duration of the customer's policy, premium payments, claim history, renewal likelihood, and potential additional services or products the customer may purchase. It helps insurers assess the long-term profitability and value associated with retaining a particular customer.

- Income - Customers income

- Monthly Premium Auto - Amount of money the customer pays on a monthly basis as a premium for their auto insurance coverage. It represents the recurring cost that the insured person must pay to maintain their insurance policy and receive coverage for potential damages, accidents, or other covered events related to their vehicle.

- Number of Open Complaints - Number of complaints the customer opened

- Policy Type - There are three type of policies in car insurance (Corporate Auto, Personal Auto, and Special Auto)

- Vehicle Class - Type of vehicle classes that customers have Two-Door Car, Four-Door Car SUV, Luxury SUV, Sports Car, and Luxury Car

- Total Claim Amount - the sum of all claims made by the customer. It represents the total monetary value of all approved claims for incidents such as accidents, theft, vandalism, or other covered events.


External Resources: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

## Challenge 1: Understanding the data

In this challenge, you will use pandas to explore a given dataset. Your task is to gain a deep understanding of the data by analyzing its characteristics, dimensions, and statistical properties.

- Identify the dimensions of the dataset by determining the number of rows and columns it contains.
- Determine the data types of each column and evaluate whether they are appropriate for the nature of the variable. You should also provide suggestions for fixing any incorrect data types.
- Identify the number of unique values for each column and determine which columns appear to be categorical. You should also describe the unique values of each categorical column and the range of values for numerical columns, and give your insights.
- Compute summary statistics such as mean, median, mode, standard deviation, and quartiles to understand the central tendency and distribution of the data for numerical columns. You should also provide your conclusions based on these summary statistics.
- Compute summary statistics for categorical columns and providing your conclusions based on these statistics.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
df = pd.read_csv(url)

# 1. Dimensions
print("Rows, columns:", df.shape)

# 2. Data types
print("\nData types:")
print(df.dtypes)

# 3. Unique values and missing values per column
print("\nUnique values per column:")
print(df.nunique())
print("\nMissing values per column:")
print(df.isna().sum())

categorical = ["ST", "GENDER", "Education", "Number of Open Complaints", "Policy Type", "Vehicle Class"]
for col in categorical:
    print(f"\n{col}: {df[col].dropna().unique().tolist()}")

numeric = df.select_dtypes(include="number")
print("\nRange of numerical columns:")
print(numeric.agg(["min", "max"]))

# 4. Summary statistics (numerical)
print("\nSummary statistics:")
print(numeric.describe())
print("\nMedian:\n", numeric.median())
print("\nMode:\n", numeric.mode().iloc[0])

# 5. Summary statistics (categorical)
print("\nCategorical summary:")
print(df[categorical].describe())

Rows, columns: (4008, 11)

Data types:
Customer                         str
ST                               str
GENDER                           str
Education                        str
Customer Lifetime Value          str
Income                       float64
Monthly Premium Auto         float64
Number of Open Complaints        str
Policy Type                      str
Vehicle Class                    str
Total Claim Amount           float64
dtype: object

Unique values per column:
Customer                     1071
ST                              8
GENDER                          5
Education                       6
Customer Lifetime Value      1027
Income                        774
Monthly Premium Auto          132
Number of Open Complaints       6
Policy Type                     3
Vehicle Class                   6
Total Claim Amount            761
dtype: int64

Missing values per column:
Customer                     2937
ST                           2937
GENDER                       3

### Conclusions Challenge 1

- The dataset has 4,008 rows and 11 columns, but only 1,071 rows contain real data: the rest are completely empty rows (all columns NaN) and should be dropped before any serious analysis.
- Several data types are wrong: `Customer Lifetime Value` is text because of the `%` sign, and `Number of Open Complaints` is text in an odd format (`1/0/00`). Both should be cleaned and converted to numeric types.
- `ST` has inconsistent labels (`California`/`Cali`, `Washington`/`WA`, `Arizona`/`AZ`) and `GENDER` has 5 variants for what should be 2-3 categories. They need to be standardized.
- Categorical columns: `ST`, `GENDER`, `Education`, `Policy Type`, `Vehicle Class` and `Number of Open Complaints`.
- `Income` is fairly symmetric (mean about 39,296 vs median about 36,234, with many customers at 0), while `Monthly Premium Auto` (mean 193 vs median 83, max 35,354) and `Total Claim Amount` (mean 405 vs median 355) are skewed to the right and contain outliers; the premium maximum looks like a data error.
- Personal Auto is by far the most frequent policy type and the categorical summaries show a few dominant categories.

## Challenge 2: analyzing the data

### Exercise 1

The marketing team wants to know the top 5 less common customer locations. Create a pandas Series object that contains the customer locations and their frequencies, and then retrieve the top 5 less common locations in ascending order.

In [ ]:
location_counts = df["ST"].value_counts()
bottom_5 = location_counts.sort_values(ascending=True).head(5)
bottom_5

ST
AZ             25
WA             30
Washington     81
Nevada         98
Cali          120
Name: count, dtype: int64

**Result:** `AZ`, `WA`, `Washington`, `Nevada` and `Cali` are the 5 least common labels. Note that `AZ`, `WA` and `Cali` are duplicates of Arizona, Washington and California, so the real ranking changes after cleaning.

### Exercise 2

The sales team wants to know the total number of policies sold for each type of policy. Create a pandas Series object that contains the policy types and their total number of policies sold, and then retrieve the policy type with the highest number of policies sold.

*Hint:*
- *Using value_counts() method simplifies this analysis.*
- *Futhermore, there is a method that returns the index of the maximum value in a column or row.*


In [ ]:
policy_counts = df["Policy Type"].value_counts()
print(policy_counts)
print("Most sold policy type:", policy_counts.idxmax())

Policy Type
Personal Auto     780
Corporate Auto    234
Special Auto       57
Name: count, dtype: int64
Most sold policy type: Personal Auto


**Result:** Personal Auto is the most sold policy type (780 policies).

### Exercise 3

The sales team wants to know if customers with Personal Auto have a lower income than those with Corporate Auto. How does the average income compare between the two policy types?

- Use *loc* to create two dataframes: one containing only Personal Auto policies and one containing only Corporate Auto policies.
- Calculate the average income for each policy.
- Print the results.

In [ ]:
personal = df.loc[df["Policy Type"] == "Personal Auto"]
corporate = df.loc[df["Policy Type"] == "Corporate Auto"]

print("Average income Personal Auto:", round(personal["Income"].mean(), 2))
print("Average income Corporate Auto:", round(corporate["Income"].mean(), 2))

Average income Personal Auto: 38180.7
Average income Corporate Auto: 41390.31


**Result:** Personal Auto customers have a lower average income (about 38,181) than Corporate Auto customers (about 41,390).

### Bonus: Exercise 4


Your goal is to identify customers with a high policy claim amount.

Instructions:

- Review again the statistics for total claim amount to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount data. 

*Note: When analyzing data, we often want to focus on certain groups of values to gain insights. Percentiles are a useful tool to help us define these groups. A percentile is a measure that tells us what percentage of values in a dataset are below a certain value. For example, the 75th percentile represents the value below which 75% of the data falls. Similarly, the 25th percentile represents the value below which 25% of the data falls. When we talk about the top 25%, we are referring to the values that fall above the 75th percentile, which represent the top quarter of the data. On the other hand, when we talk about the bottom 25%, we are referring to the values that fall below the 25th percentile, which represent the bottom quarter of the data. By focusing on these groups, we can identify patterns and trends that may be useful for making decisions and taking action.*

*Hint: look for a method that gives you the percentile or quantile 0.75 and 0.25 for a Pandas Series.*

*Hint 2: check `Boolean selection according to the values of a single column` in https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9*

In [ ]:
q75 = df["Total Claim Amount"].quantile(0.75)
print("75th percentile of Total Claim Amount:", q75)

high_claim = df.loc[df["Total Claim Amount"] > q75]
print("High-claim customers:", len(high_claim))

# Summary statistics of the high-claim group
high_claim.describe()

75th percentile of Total Claim Amount: 532.8
High-claim customers: 264


             Income  Monthly Premium Auto  Total Claim Amount
count    264.000000            264.000000          264.000000
mean   23677.344697            165.193182          782.228263
std    27013.483721            623.930992          292.751640
min        0.000000             63.000000          537.600000
25%        0.000000             99.000000          606.521741
50%    18807.000000            114.000000          679.597985
75%    42423.750000            133.250000          851.400000
max    99316.000000          10202.000000         2893.239678